# Lab proof · Job Match Studio

One app that merges two tools built in Claude chat:

| Source | What it contributes |
|---|---|
| **Lovable app** *Job Search & Match Score Pro* | Board, Analysis and Application Materials layout, Fit/Win gauge cards, pipeline stages, canonical facts and drafting rules |
| **Claude artifact** *Job Match Board* | The scoring logic: candidate profile, outcome calibration, Fit vs Win rubric, offline keyword scorer, LinkedIn launcher, scored history |
| **New in this project** | A RAG layer over the Google Drive project folder, outcome-calibrated few-shot prompting, rubric guardrails, a Gradio UI |

This notebook runs **every step of the pipeline on real data** and records a pass/fail check for each one. The last cell writes `quality_check/LAB_PROOF.md`.

```
Google Drive folder ──drive_sync──▶ data/drive (mirror + manifest)
        │ ingest: parse → canonicalise → section-aware chunks → contextual headers → tags
        ▼
 Chroma (local bge embeddings)  +  BM25
        │ per requirement: dense×2 + BM25×2 (wording + HyDE query) → RRF → Cohere rerank → authority prior → dedupe → budget
        ▼
 posting ─▶ decompose (LLM) ─▶ evidence pack [E#] ─▶ few-shot: 3 nearest past jobs + real outcomes ─▶ rubric prompt
        ─▶ Structured Output (evidence first, scores last) ─▶ guardrails + sanitiser ─▶ SQLite board ─▶ Gradio UI
                                                                └─▶ materials: CV + cover letter ─▶ 11 checks ─▶ self-repair ─▶ .docx
```

In [1]:
import json, os, platform, re, statistics, sys, time, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
ROOT = Path.cwd() if (Path.cwd() / "config.py").exists() else Path.cwd().parent
os.chdir(ROOT); sys.path.insert(0, str(ROOT))
import numpy as np, pandas as pd
from IPython.display import HTML, Markdown, display
pd.set_option("display.max_colwidth", 110); pd.set_option("display.width", 220)
import config, candidate as C, ingest, rag, assess, scoring, llm, store, materials, drive_sync, ui

CHECKS = []
def check(step, name, ok, detail=""):
    CHECKS.append({"Step": step, "Check": name, "Result": "PASS" if ok else "FAIL", "Evidence": str(detail)[:200]})
    print(("✅ " if ok else "❌ ") + name + (f"  ({detail})" if detail != "" else ""))
    return bool(ok)

def show_html(fragment):  # render app components with the app's own stylesheet
    display(HTML(f"<style>{ui.CSS}</style>{fragment}"))

RUN_STARTED = time.time()
print("Project:", ROOT.name, "| run:", time.strftime("%Y-%m-%d %H:%M"))

Project: job-search-score-apply-bot | run: 2026-11-06 15:27


## Step 0 · Environment and keys
Keys are read from exported environment variables or from `~/.config/job-search-bot/.env`, outside the project folder. Their values are never printed or written.

In [2]:
import gradio, openai, chromadb, sentence_transformers, cohere
secrets = [v for v in (config.OPENAI_API_KEY, config.COHERE_API_KEY) if v]
env_file = ROOT / ".env"
stored = env_file.exists() and any(s in env_file.read_text() for s in secrets)
display(pd.DataFrame([
    ("Python", platform.python_version()), ("gradio", gradio.__version__), ("openai", openai.__version__),
    ("chromadb", chromadb.__version__), ("sentence-transformers", sentence_transformers.__version__), ("cohere", cohere.__version__),
    ("Embedding model (runs locally)", config.EMBED_MODEL), ("Reranker", rag.reranker_name()),
    ("LLM for assessment and materials", config.LLM_MODEL), ("LLM for posting decomposition", config.LLM_MODEL_FAST),
    ("OPENAI_API_KEY", "set" if config.OPENAI_API_KEY else "missing"), ("COHERE_API_KEY", "set" if config.COHERE_API_KEY else "missing"),
], columns=["Component", "Value"]))
check("0 Setup", "OpenAI key available to this run (value never printed)", llm.available())
check("0 Setup", "Cohere key available, so reranking uses rerank-v3.5", bool(config.COHERE_API_KEY), rag.reranker_name())
check("0 Setup", "Keys come from the environment, not from a file in the project", not stored)

W1106 15:27:04.910000 38394 site-packages/torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


,Component,Value
0,Python,3.11.17
1,gradio,6.31.0
2,openai,3.26.1
3,chromadb,1.5.9
4,sentence-transformers,6.1.0
5,cohere,7.2.0
6,Embedding model (runs locally),BAAI/bge-small-en-v1.5
7,Reranker,cohere
8,LLM for assessment and materials,gpt-5.6-luna
9,LLM for posting decomposition,gpt-5.4-mini


✅ OpenAI key available to this run (value never printed)
✅ Cohere key available, so reranking uses rerank-v3.5  (cohere)
✅ Keys come from the environment, not from a file in the project


True

## Step 1 · Google Drive → local mirror
The project folder *Claude Project for Job2026* is mirrored to `data/drive` with a manifest (file id, folder, modified time, link). `drive_sync.py` does this through the Drive API; this run's mirror was pulled through Claude's Google Drive connector in the same format.

In [3]:
manifest = json.loads((config.DRIVE_DIR / "manifest.json").read_text())
mf = pd.DataFrame(manifest)
mf["type"] = mf.title.map(ingest.doc_type)
mf["kind"] = mf.mimeType.str.extract(r"(document|pdf|presentation|json|markdown)", expand=False).fillna("file")
mf["chars"] = [len((config.DRIVE_DIR / p).read_text(encoding="utf-8")) for p in mf.path]
display(mf[["title", "folder", "type", "kind", "modifiedTime", "chars", "source"]])
check("1 Drive", "Core project documents mirrored from Google Drive", len(manifest) >= 10, f"{len(manifest)} documents, {mf.chars.sum():,} characters")
check("1 Drive", "Every mirrored document has text", (mf.chars > 1000).all(), f"smallest has {mf.chars.min():,} characters")
check("1 Drive", "Each document links back to its Drive file", mf.viewUrl.str.startswith("https://").all())

,title,folder,type,kind,modifiedTime,chars,source
0,Ayhan_Demir_CV_General.docx,APPLICATION MATERIALS,cv,document,2026-08-06T13:26:08Z,8517,claude-drive-connector
1,Ayhan_Demir_Phase1A_Case_Studies_FINAL_June2026.docx,"Phase2 - Main CV, All Portfolio & Json file",case_study,document,2026-06-24T14:49:34Z,29057,claude-drive-connector
2,Ayhan_Demir_Phase1B_Case_Studies_FINAL_June2026.docx,"Phase2 - Main CV, All Portfolio & Json file",case_study,document,2026-06-24T14:49:21Z,31954,claude-drive-connector
3,Phase0 - Portfolio Strategic Map - Final Version_June2026.docx,"Phase2 - Main CV, All Portfolio & Json file",strategy_map,document,2026-06-24T14:49:52Z,37429,claude-drive-connector
4,Ayhan_Demir_Amazon_Career_Narrative.docx,Wolt Sr.Program Manager,career_narrative,document,2026-09-23T13:14:41Z,3778,claude-drive-connector
5,Ayhan_Demir_Wolt_Dan_CoreProjects.docx,Wolt Sr.Program Manager,interview_prep,document,2026-09-23T13:15:30.785Z,35771,claude-drive-connector
6,Ayhan_Demir_Wolt_35_HM_Questions_Concise,Wolt Sr.Program Manager,interview_prep,document,2026-10-20T18:15:31.054Z,20083,claude-drive-connector
7,Ayhan_Demir_Digital_Tools_OnePager.docx,PRODUCTS: Agentic AI,one_pager,document,2026-08-06T13:47:27Z,1924,claude-drive-connector
8,Ayhan_Demir_Consulting_OnePagers_2026-07-28.docx,Phase10 - Consultancy Side Hustle,one_pager,document,2026-07-28T18:46:29Z,5722,claude-drive-connector
9,Ayhan_Demir_CoverLetter_Accenture_AITransformation.docx,APPLICATION MATERIALS,cover_letter,document,2026-08-06T13:49:09Z,2572,claude-drive-connector


✅ Core project documents mirrored from Google Drive  (11 documents, 179,226 characters)
✅ Every mirrored document has text  (smallest has 1,924 characters)
✅ Each document links back to its Drive file


True

### Step 1b · Sync selection logic
The folder holds about 90 files, many of them older versions, dashboard code, images and backups. `drive_sync.select()` keeps only the newest version of each document. Tested here on real titles and dates from the folder listing.

In [4]:
listing = [
    ("Ayhan_Demir_Phase1A_Case_Studies_FINAL_June2026.docx", "docx", "2026-06-24T14:49:34Z", 21792),
    ("Ayhan_Demir_Phase1A_Case_Studies_FINAL.docx", "docx", "2026-05-13T15:09:38Z", 21536),
    ("Ayhan_Demir_Phase1A_Case_Studies_v4- ayhan review", "gdoc", "2026-05-13T14:54:57Z", 30652),
    ("Ayhan_Demir_Wolt_35_HM_Questions_Concise", "gdoc", "2026-10-20T18:15:31Z", 14803),
    ("Ayhan_Demir_Wolt_35_HM_Questions_Concise.docx", "docx", "2026-09-23T17:37:28Z", 18230),
    ("Ayhan_Demir_Portfolio_Data_July_2026_Final_Version.json", "json", "2026-07-22T14:00:03Z", 109840),
    ("Ayhan_Demir_Portfolio_Data_2026-07-03.json", "json", "2026-07-03T14:12:42Z", 109730),
    ("Ayhan_Demir_Assignment_Defect_Elimination_Program_backup_copy.pptx", "pptx", "2026-09-29T15:12:54Z", 4906610),
    ("Ayhan_Demir_Phase1B_Case_Studies_FINAL_ayhan review dosyasının kopyası", "gdoc", "2026-05-13T19:42:29Z", 22849),
    ("CV_Photo.PNG", "png", "2026-06-24T14:56:55Z", 1356938),
    ("Ayhan_Demir_Portfolio_Dashboard_v2.jsx", "jsx", "2026-06-24T17:17:48Z", 151078),
    ("Reference letter English.pdf", "pdf", "2026-06-24T14:50:49Z", 30056669),
]
MIME = {"docx": "application/vnd.openxmlformats-officedocument.wordprocessingml.document", "gdoc": "application/vnd.google-apps.document",
        "json": "application/json", "pptx": "application/vnd.openxmlformats-officedocument.presentationml.presentation",
        "png": "image/png", "jsx": "application/octet-stream", "pdf": "application/pdf"}
files = [{"id": str(i), "name": n, "mimeType": MIME[k], "modifiedTime": t, "size": str(s), "folder": ""} for i, (n, k, t, s) in enumerate(listing)]
kept = {f["name"] for f in drive_sync.select(files)}
display(pd.DataFrame([{"file": n, "version group": drive_sync.stem(n), "kept": "yes" if n in kept else ""} for n, *_ in listing]))
newest = {"Ayhan_Demir_Phase1A_Case_Studies_FINAL_June2026.docx", "Ayhan_Demir_Wolt_35_HM_Questions_Concise",
          "Ayhan_Demir_Portfolio_Data_July_2026_Final_Version.json"}
check("1 Drive", "Sync keeps only the newest version of each document", kept == newest, f"{len(kept)} of {len(listing)} kept")
check("1 Drive", "Sync skips backups, copies, images, code exports and oversized scans",
      not kept & {"CV_Photo.PNG", "Reference letter English.pdf", "Ayhan_Demir_Portfolio_Dashboard_v2.jsx"})

,file,version group,kept
0,Ayhan_Demir_Phase1A_Case_Studies_FINAL_June2026.docx,phase1a case studies,yes
1,Ayhan_Demir_Phase1A_Case_Studies_FINAL.docx,phase1a case studies,
2,Ayhan_Demir_Phase1A_Case_Studies_v4- ayhan review,phase1a case studies,
3,Ayhan_Demir_Wolt_35_HM_Questions_Concise,wolt 35 hm questions concise,yes
4,Ayhan_Demir_Wolt_35_HM_Questions_Concise.docx,wolt 35 hm questions concise,
5,Ayhan_Demir_Portfolio_Data_July_2026_Final_Version.json,portfolio data,yes
6,Ayhan_Demir_Portfolio_Data_2026-07-03.json,portfolio data,
7,Ayhan_Demir_Assignment_Defect_Elimination_Program_backup_copy.pptx,assignment defect elimination program backup copy,
8,Ayhan_Demir_Phase1B_Case_Studies_FINAL_ayhan review dosyasının kopyası,phase1b case studies dosyas n n kopyas,
9,CV_Photo.PNG,cv photo png,


✅ Sync keeps only the newest version of each document  (3 of 12 kept)
✅ Sync skips backups, copies, images, code exports and oversized scans


True

## Step 2 · Parse and canonicalise
The canonical record outranks the source documents. Mentions of the excluded project are removed at the smallest unit: a sentence in prose, a row in a table, or a whole table when its title row is about it. Titles the record forbids are rewritten. Retrieval can therefore never surface a claim you have ruled out.

In [5]:
t = time.time(); report = ingest.run(); build_s = round(time.time() - t, 1)
docs = pd.DataFrame(report["docs"])
display(docs[["title", "type", "chunks", "tokens", "redacted_blocks", "replaced"]])
chunks = [json.loads(line) for line in config.CHUNKS_PATH.open(encoding="utf-8")]
phase0 = ingest.clean(ingest.load_text(config.DRIVE_DIR / "Phase0_Portfolio_Strategic_Map_Final_Version_June2026.md"))
excluded = re.compile("|".join(C.EXCLUDED_CLAIMS), re.I)
line = next(l for l in phase0.split("\n") if "VP WW Global" in l)
print("Before:", line[:150]); print("After: ", ingest.canonicalize(line)[0][:150])
step3 = sum(c["section"].startswith("STEP 3") for c in chunks)
check("2 Canonicalise", "Excluded project (ACES AI Strategy) removed from the index", not any(excluded.search(c["text"]) for c in chunks),
      f"{len(excluded.findall(phase0))} mentions in the Phase 0 map, 0 in the index")
check("2 Canonicalise", "Canonical title enforced (no 'VP' for the wellbeing role)", not any("VP WW Global" in c["text"] for c in chunks), f"{docs.replaced.sum()} rewrites")
check("2 Canonicalise", "Redaction is surgical: the 60-row project map survives", step3 >= 5, f"{step3} project-map chunks kept")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 11236.61it/s]

,title,type,chunks,tokens,redacted_blocks,replaced
0,Ayhan_Demir_CV_General.docx,cv,11,1794,0,1
1,Ayhan_Demir_Phase1A_Case_Studies_FINAL_June2026.docx,case_study,29,5558,0,0
2,Ayhan_Demir_Phase1B_Case_Studies_FINAL_June2026.docx,case_study,29,5741,0,4
3,Phase0 - Portfolio Strategic Map - Final Version_June2026.docx,strategy_map,37,9755,9,5
4,Ayhan_Demir_Amazon_Career_Narrative.docx,career_narrative,3,787,0,0
5,Ayhan_Demir_Wolt_Dan_CoreProjects.docx,interview_prep,35,7256,0,0
6,Ayhan_Demir_Wolt_35_HM_Questions_Concise,interview_prep,36,3603,0,0
7,Ayhan_Demir_Digital_Tools_OnePager.docx,one_pager,5,340,0,0
8,Ayhan_Demir_Consulting_OnePagers_2026-07-28.docx,one_pager,5,976,1,0
9,Ayhan_Demir_CoverLetter_Accenture_AITransformation.docx,cover_letter,3,501,0,0


Before: | 2021 | L5 | CS MHW (Mental Health Wellbeing) Affinity Group, Founded, grew to VP WW Global | P5 | Strong Evidence |
After:  | 2021 | L5 | CS MHW (Mental Health Wellbeing) Affinity Group, Founded, grew to a Worldwide programme | P5 | Strong Evidence |
✅ Excluded project (ACES AI Strategy) removed from the index  (16 mentions in the Phase 0 map, 0 in the index)
✅ Canonical title enforced (no 'VP' for the wellbeing role)  (10 rewrites)
✅ Redaction is surgical: the 60-row project map survives  (11 project-map chunks kept)


True

## Step 3 · Section-aware chunking
Chunks follow the documents' own structure (case study › subsection, interview question, table rows with their header repeated). Each chunk is embedded with a contextual header, and tagged with signature projects and pillars.

In [6]:
toks = [c["tokens"] for c in chunks]
display(pd.DataFrame([{"documents": len(docs), "chunks": len(chunks), "min tokens": min(toks), "median tokens": statistics.median(toks),
                       "max tokens": max(toks), "build seconds (parse+embed+index)": build_s}]))
display(pd.Series([c["doc_type"] for c in chunks]).value_counts().rename_axis("document type").reset_index(name="chunks"))
sample = next(c for c in chunks if c["section"].startswith("Case Study 02") and c["subsection"] == "Results")
print(sample["context"]); print("projects:", sample["projects"], "| pillars:", sample["pillars"], "| tokens:", sample["tokens"], "\n")
print(sample["text"][:520])
table_piece = next(c for c in chunks if c["section"].startswith("STEP 3") and c["text"].startswith("| Year"))
print("\nA later piece of the project-map table still starts with its header row:\n" + table_piece["text"][:260])
tagged = sum(bool(c["projects"]) for c in chunks)
check("3 Chunking", f"Chunks fit the {config.CHUNK_TOKENS}-token budget", max(toks) <= config.CHUNK_TOKENS, f"max {max(toks)}, median {statistics.median(toks)}")
check("3 Chunking", "Every chunk has a contextual header and a Drive link", all(c["context"].startswith("[") and c["url"] for c in chunks))
check("3 Chunking", "Case-study structure preserved (document › case study › subsection)", " › Case Study 02" in sample["context"] and sample["context"].endswith("Results]"))
check("3 Chunking", "Table pieces repeat the header row", table_piece["text"].startswith("| Year | Level | Project |"))
check("3 Chunking", "Most chunks tagged with a signature project", tagged >= len(chunks) / 2, f"{tagged} of {len(chunks)}")

,documents,chunks,min tokens,median tokens,max tokens,build seconds (parse+embed+index)
0,11,195,8,195,320,4.6


,document type,chunks
0,interview_prep,71
1,case_study,58
2,strategy_map,37
3,cv,11
4,one_pager,10
5,cover_letter,5
6,career_narrative,3


[Ayhan Demir Phase1A Case Studies FINAL June2026 › Case Study 02 · Concession Permission Clean-Up (EU7) · 2023 · P2 · Portfolio Design and Delivery › Results]
projects: Concession Permission Clean-Up | pillars: p3|p2 | tokens: 212 

$132M GCV (Gross Concession Value) reduction — finance-validated

$20.06M annualised net savings confirmed

40.03% to 23.97% over-permissioned agents across EU7: 16 percentage points closed

1,225 LDAP groups covered and remediated across EU7

Approximately 52% of the full 120-person ACES team's 2023 total savings of $253M came from this single project

Once CS Award received · L6 promotion confirmation followed within months of project close

Three structural access controls remain in production — a permanent gove

A later piece of the project-map table still starts with its header row:
| Year | Level | Project | Pillar | Evidence |
| 2019 | L4 | Shipment Confirmation / View Invoice Self-Service | P2 | Table Only |

| Year | Level | Project | Pillar | Evid

True

## Step 4 · Embeddings and vector database
`BAAI/bge-small-en-v1.5` runs locally, so your documents never leave the machine for indexing. Vectors are stored in a persistent Chroma collection (cosine), with metadata for filtering.

In [7]:
r = rag.get()
mat = np.stack([r.vec[c["id"]] for c in chunks])
display(pd.DataFrame([{"collection": config.COLLECTION, "vectors": r.col.count(), "dimension": mat.shape[1],
                       "distance": r.col.metadata.get("hnsw:space"), "mean vector norm": round(float(np.linalg.norm(mat, axis=1).mean()), 4)}]))
check("4 Vector DB", "Chroma holds one vector per chunk", r.col.count() == len(chunks), f"{r.col.count()} vectors")
check("4 Vector DB", "Vectors are normalised for cosine search", bool(np.allclose(np.linalg.norm(mat, axis=1), 1, atol=1e-3)))

,collection,vectors,dimension,distance,mean vector norm
0,portfolio_evidence,195,384,cosine,1.0


✅ Chroma holds one vector per chunk  (195 vectors)
✅ Vectors are normalised for cosine search


True

## Step 5 · One query through every retrieval stage

In [8]:
q = "influencing partner teams without formal authority to prioritise a fix"
trace = {}; final = r.search(q, k=5, trace=trace)
label = lambda cid: f"{r.by_id[cid]['doc_title'][:24]} · {(r.by_id[cid]['section'] or r.by_id[cid]['subsection'] or 'main')[:36]}"
stages = pd.DataFrame({name: [label(cid) for cid, _ in trace[name][:5]] for name in ("dense", "bm25", "fused", "reranked")}, index=range(1, 6))
display(stages)
print("Reranker scores:", [round(s, 3) for _, s in trace["reranked"]])
check("5 Retrieval", "Dense, BM25, fusion and rerank stages all return results", all(len(trace[s]) for s in ("dense", "bm25", "fused", "reranked")))

,dense,bm25,fused,reranked
1,Ayhan Demir Wolt Dan Cor · Question 3: Getting a team you don't,Ayhan Demir Wolt Dan Cor · Question 3: Getting a team you don't,Ayhan Demir Wolt Dan Cor · Question 3: Getting a team you don't,Ayhan Demir Wolt Dan Cor · Question 3: Getting a team you don't
2,Ayhan Demir Wolt 35 HM Q · 19. Getting partner teams to accept,Ayhan Demir Wolt Dan Cor · Question 3: Getting a team you don't,Ayhan Demir Wolt 35 HM Q · 19. Getting partner teams to accept,Ayhan Demir Wolt Dan Cor · Question 3: Getting a team you don't
3,Ayhan Demir Wolt Dan Cor · Question 3: Getting a team you don't,Ayhan Demir Wolt Dan Cor · Question 3: Getting a team you don't,Ayhan Demir Phase1A Case · Case Study 03 · VSO Programme and IX,Ayhan Demir Wolt Dan Cor · Question 3: Getting a team you don't
4,Ayhan Demir Wolt 35 HM Q · 17. Resolving a conflict or disagre,Ayhan Demir Phase1A Case · Case Study 03 · VSO Programme and IX,Ayhan Demir Wolt Dan Cor · Question 3: Getting a team you don't,Ayhan Demir Wolt Dan Cor · Question 3: Getting a team you don't
5,Ayhan Demir Wolt Dan Cor · Question 3: Getting a team you don't,Ayhan Demir Wolt 35 HM Q · 19. Getting partner teams to accept,Ayhan Demir Wolt Dan Cor · Question 3: Getting a team you don't,Ayhan Demir Wolt 35 HM Q · 19. Getting partner teams to accept


Reranker scores: [0.724, 0.71, 0.705, 0.623, 0.601]
✅ Dense, BM25, fusion and rerank stages all return results


True

## Step 6 · Retrieval evaluation
There are 21 queries, each labelled with the signature project the answer must come from. Twelve are keyword-rich, using terms like IMEI or Kaizen. Nine are paraphrased with no project vocabulary, the way a job posting would phrase the need. A hit counts when the chunk is tagged with that project and is focused on it (at most 3 projects tagged). Five configurations are compared on each set.

In [9]:
EVAL = {"Keyword-rich": [
    ("GenAI audit system validated against 10,000 contacts", "AuditAI"),
    ("cut a manual quality audit cycle from seven days to two", "AuditAI"),
    ("clean-up of over-permissioned agents holding concession authority", "Concession Permission Clean-Up"),
    ("no automated deprovisioning when stretch assignments ended", "Concession Permission Clean-Up"),
    ("first end-to-end value stream offsite with 25 cross-functional stakeholders", "VSO / IXD-FC"),
    ("nobody owned packing, a systemic hidden cost", "VSO / IXD-FC"),
    ("Kaizen that fixed the activation SLA of the backup specialist group", "BUWG Kaizen"),
    ("operating model with a technical bar and development plans adopted across seven European markets", "ACES Value Offers Framework"),
    ("five-phase contact reduction with ML recommenders and an email to phone channel shift", "PLS Multi-Phase Programme"),
    ("iPhone return fraud stopped with IMEI blocking and chain-of-custody pickup", "Mail Label Fraud"),
    ("founded an employee mental health peer support group and trained its board", "Mental Health & Wellbeing Group"),
    ("design of experiments that cut paintshop rework in a washing machine plant", "Arcelik Foundation"),
], "Paraphrased": [
    ("an LLM tool that scores the quality of customer conversations", "AuditAI"),
    ("staff kept elevated system rights long after temporary roles ended", "Concession Permission Clean-Up"),
    ("workshop mapping how dozens of improvement projects connect along the customer journey", "VSO / IXD-FC"),
    ("overflow specialists were slow to be brought in; fixed the hand-offs instead of hiring", "BUWG Kaizen"),
    ("defined what the improvement team offers the business and how its people grow", "ACES Value Offers Framework"),
    ("each fix revealed the next root cause of avoidable customer calls", "PLS Multi-Phase Programme"),
    ("stopped returns abuse on high-value phones together with the manufacturer and a courier", "Mail Label Fraud"),
    ("built a peer support network for colleagues under stress and measured the effect", "Mental Health & Wellbeing Group"),
    ("built every quality lab and inspection process for a new factory from scratch", "Arcelik Foundation"),
]}
def relevant(chunk, project):
    tags = [p for p in chunk["projects"].split("|") if p]
    return project in tags and len(tags) <= 3

def evaluate(queries, mode, reranker=None):
    if reranker:
        r.reranker = reranker
    hit1 = hit3 = mrr = prec = 0
    for query, project in queries:
        flags = [relevant(h, project) for h in r.search(query, k=5, mode=mode)]
        hit1 += flags[0]; hit3 += any(flags[:3]); prec += sum(flags) / 5
        mrr += next((1 / (i + 1) for i, f in enumerate(flags) if f), 0)
    n = len(queries)
    return {"Hit@1": round(hit1 / n, 2), "Hit@3": round(hit3 / n, 2), "MRR@5": round(mrr / n, 3), "Precision@5": round(prec / n, 2)}

CONFIGS = [("Dense only (bge-small)", "dense", None), ("BM25 only", "bm25", None), ("Hybrid (RRF)", "hybrid", None),
           ("Hybrid + local cross-encoder", "hybrid+rerank", "local")]
if config.COHERE_API_KEY:
    CONFIGS.append(("Hybrid + Cohere rerank-v3.5", "hybrid+rerank", "cohere"))
rows = [{"Query set": name, "Configuration": label, **evaluate(queries, mode, rr)}
        for name, queries in EVAL.items() for label, mode, rr in CONFIGS]
r.reranker = rag.reranker_name()
EVAL_TABLE = pd.DataFrame(rows)
display(EVAL_TABLE.set_index(["Query set", "Configuration"]))
para = EVAL_TABLE[EVAL_TABLE["Query set"] == "Paraphrased"].set_index("Configuration")
default_cfg = "Hybrid + Cohere rerank-v3.5" if config.COHERE_API_KEY else "Hybrid + local cross-encoder"
overall = {label: sum(evaluate(q, mode, rr)["Hit@3"] * len(q) for q in EVAL.values()) / sum(len(q) for q in EVAL.values())
           for label, mode, rr in CONFIGS if label == default_cfg}
r.reranker = rag.reranker_name()
check("6 Retrieval eval", "On paraphrased queries, dense search beats BM25 (semantic search earns its place)",
      para.loc["Dense only (bge-small)", "MRR@5"] > para.loc["BM25 only", "MRR@5"],
      f"MRR@5 {para.loc['Dense only (bge-small)', 'MRR@5']} vs {para.loc['BM25 only', 'MRR@5']}")
check("6 Retrieval eval", f"The app's default ({default_cfg}) has the best MRR@5 on paraphrased queries",
      para.loc[default_cfg, "MRR@5"] >= para["MRR@5"].max(), f"{para.loc[default_cfg, 'MRR@5']}")
check("6 Retrieval eval", "The default finds the right project in the top 3 for at least 90% of all 21 queries",
      overall[default_cfg] >= 0.9, f"Hit@3 {round(overall[default_cfg], 2)}")

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 9754.63it/s]

Hit@1  Hit@3  MRR@5  Precision@5
Query set    Configuration                                                 
Keyword-rich Dense only (bge-small)         0.92   1.00  0.958         0.92
             BM25 only                      1.00   1.00  1.000         0.98
             Hybrid (RRF)                   1.00   1.00  1.000         0.98
             Hybrid + local cross-encoder   1.00   1.00  1.000         1.00
             Hybrid + Cohere rerank-v3.5    1.00   1.00  1.000         1.00
Paraphrased  Dense only (bge-small)         0.89   1.00  0.944         0.84
             BM25 only                      0.78   1.00  0.870         0.80
             Hybrid (RRF)                   0.89   1.00  0.926         0.84
             Hybrid + local cross-encoder   0.89   0.89  0.917         0.84
             Hybrid + Cohere rerank-v3.5    1.00   1.00  1.000         0.91

✅ On paraphrased queries, dense search beats BM25 (semantic search earns its place)  (MRR@5 0.944 vs 0.87)
✅ The app's default (Hybrid + Cohere rerank-v3.5) has the best MRR@5 on paraphrased queries  (1.0)
✅ The default finds the right project in the top 3 for at least 90% of all 21 queries  (Hit@3 1.0)


True

## Step 7 · Posting decomposition (LLM, structured output)
The test posting is your real Wolt *Senior Program Manager, CX and Defect Elimination* posting from Drive. You reached the final round for it, which makes it a calibration case. The fast model turns it into facts plus prioritised requirements, each with a HyDE-style evidence query written like a CV line.

In [10]:
wolt = (config.POSTINGS_DIR / "Wolt_JobPosting_SeniorPM_DefectElimination.md").read_text()
t = time.time(); posting = assess.decompose(wolt); dec_s = round(time.time() - t, 1)
print(f"{posting.title} · {posting.company} · {posting.location} | level: {posting.level} | German required: {posting.german_required} | screens: {posting.screens or 'none'} | {dec_s}s")
display(pd.DataFrame([q.model_dump() for q in posting.requirements]).set_index("id"))
check("7 Decompose", "Posting broken into 4 to 7 prioritised requirements", 4 <= len(posting.requirements) <= 7, f"{len(posting.requirements)} requirements")
check("7 Decompose", "Each requirement has a HyDE-style evidence query", all(len(q.evidence_query.split()) >= 6 for q in posting.requirements))
check("7 Decompose", "No German requirement found in an English-only posting", not posting.german_required)

Senior Program Manager, Customer Experience and Defect Elimination · Wolt · Berlin, Germany | level: Senior | German required: False | screens: none | 4.1s


,ask,pillar,importance,evidence_query
id,,,,
R1,Own and drive a large-scale customer experience / defect elimination program end to end across multiple ma...,P2,critical,"Led a multi-market customer experience or operations program end to end, owning roadmap, prioritisation, d..."
R2,Translate customer defects into root-cause analysis and systemic fixes with partner teams.,Domain,critical,"Identified customer defects, performed root cause analysis, and drove systemic fixes through cross-functio..."
R3,"Use voice-of-the-customer and analytical insight to build a credible, prioritised case from imperfect data.",P3,critical,Built a voice-of-the-customer or customer insight program using imperfect or directional data to quantify ...
R4,"Influence engineering, operations, and finance stakeholders to prioritise customer-impacting work and comm...",P4,critical,"Partnered with engineering, operations, and finance stakeholders to secure prioritisation of customer-impa..."
R5,"Present clear, actionable customer-experience insights in business reviews, planning sessions, and leaders...",P1,important,"Presented customer experience insights and recommendations in product reviews, business reviews, or planni..."
R6,Maintain and improve reporting or data infrastructure that supports actionable customer experience managem...,P3,important,"Built or improved reporting, dashboards, or data infrastructure that made customer experience insights cre..."


✅ Posting broken into 4 to 7 prioritised requirements  (6 requirements)
✅ Each requirement has a HyDE-style evidence query
✅ No German requirement found in an English-only posting


True

## Step 8 · Evidence pack
For each requirement, retrieval runs on both its wording and its HyDE query, with dense and BM25 search. Results are fused with RRF, reranked against the requirement, and given a small boost for primary documents (CV, case studies) over restatements. Near-duplicates are dropped, each document is capped, and the pack is filled up to a token budget. Cover letters are excluded because they echo the posting they were written for.

In [11]:
t = time.time(); pack = r.evidence_pack([q.model_dump() for q in posting.requirements]); pack_s = round(time.time() - t, 1)
display(pd.DataFrame(pack)[["eid", "supports", "doc_title", "doc_type", "section", "score", "tokens"]])
critical = {q.id for q in posting.requirements if q.importance == "critical"}
covered = {s for e in pack for s in e["supports"]}
check("8 Evidence", "Every critical requirement has retrieved evidence", critical <= covered, f"{len(critical)} critical requirements" if critical <= covered else f"missing {critical - covered}")
check("8 Evidence", f"Evidence fits the {config.EVIDENCE_BUDGET}-token budget", sum(e["tokens"] for e in pack) <= config.EVIDENCE_BUDGET,
      f"{sum(e['tokens'] for e in pack)} tokens in {len(pack)} chunks, {pack_s}s")
check("8 Evidence", "Cover letters excluded from evidence (style examples only)", all(e["doc_type"] != "cover_letter" for e in pack))
check("8 Evidence", "Evidence drawn from several documents, including primary ones", len({e["doc_title"] for e in pack}) >= 3 and any(e["doc_type"] in config.PRIMARY_TYPES for e in pack),
      f"{len({e['doc_title'] for e in pack})} documents")

,eid,supports,doc_title,doc_type,section,score,tokens
0,E1,"[R1, R2, R3, R4]",Ayhan Demir Wolt 35 HM Questions Concise,interview_prep,4. What do you understand this role to be?,0.697,93
1,E2,[R1],Ayhan Demir Amazon Career Narrative,career_narrative,,0.480,269
2,E3,[R2],Ayhan Demir Wolt 35 HM Questions Concise,interview_prep,19. Getting partner teams to accept work that competes with their priorities.,0.470,95
3,E4,[R3],Ayhan Demir Wolt Dan CoreProjects,interview_prep,Question 2: Tell me about working with imperfect or directional data,0.350,240
4,E5,[R4],Phase0 - Portfolio Strategic Map - Final Version June2026,strategy_map,STEP 2 — Taxonomy Schema: 5 Strategic Pillars,0.190,265
5,E6,[R5],Ayhan Demir CV General,cv,,0.081,289
6,E7,"[R5, R6]",Phase0 - Portfolio Strategic Map - Final Version June2026,strategy_map,STEP 7 — Acronym Master List,0.052,298
7,E8,[R6],Phase0 - Portfolio Strategic Map - Final Version June2026,strategy_map,STEP 7 — Acronym Master List,0.091,222


✅ Every critical requirement has retrieved evidence  (4 critical requirements)
✅ Evidence fits the 3200-token budget  (1771 tokens in 8 chunks, 9.9s)
✅ Cover letters excluded from evidence (style examples only)
✅ Evidence drawn from several documents, including primary ones  (5 documents)


True

## Step 9 · Dynamic few-shot with real outcomes
The three most similar past assessments from your artifact board are shown to the model with their scores and what actually happened. Leave-one-out: the model never sees its own earlier verdict on this exact role.

In [12]:
examples = assess.pick_examples(posting)
display(pd.DataFrame(examples)[["title", "company", "fit", "win", "evidence", "outcome", "similarity"]])
check("9 Few-shot", "Three outcome-labelled examples selected by similarity", len(examples) == 3 and all(e["outcome"] for e in examples))
check("9 Few-shot", "Leave-one-out: the same Wolt role is never shown to the model",
      not any(assess._same_role({"title": e["title"], "company": e["company"]}, posting) for e in examples))

,title,company,fit,win,evidence,outcome,similarity
0,"Manager, Merchant Operations",Wolt,70,52,adjacent,"applied, rejected (stage reached not recorded)",0.742
1,"S&O Manager, Support Tooling and Systems",Wolt (DoorDash / Deliveroo),62,33,adjacent,"applied, rejected (stage reached not recorded)",0.734
2,"Customer Experience Strategy Lead (m,f,x)",HelloFresh,74,46,direct,"still live, at recruiter screen",0.731


✅ Three outcome-labelled examples selected by similarity
✅ Leave-one-out: the same Wolt role is never shown to the model


True

## Step 10 · Prompt assembly

In [13]:
system, user = assess.build_messages(wolt, posting, pack, examples)
parts = {"Profile (artifact)": C.PROFILE, "Outcome calibration (artifact)": C.OUTCOME_CALIBRATION, "Fit/Win rubric (artifact)": C.RUBRIC,
         "Few-shot examples": assess.format_examples(examples), "Requirements": "\n".join(q.ask for q in posting.requirements),
         "Evidence pack": "\n".join(e["text"] for e in pack), "Job text": wolt}
display(pd.DataFrame([{"Prompt section": k, "Tokens": ingest.ntok(v)} for k, v in parts.items()]))
print(user[:1300], "…")
total = ingest.ntok(system + user)
check("10 Prompt", "Prompt carries calibration, examples, requirements, cited evidence and the job text",
      all(s in system + user for s in ("OUTCOME CALIBRATION", "CALIBRATION EXAMPLES", "EVIDENCE PACK", "JOB TEXT", "[E1]")))
check("10 Prompt", "Prompt stays compact", total < 16000, f"{total:,} tokens")

,Prompt section,Tokens
0,Profile (artifact),1059
1,Outcome calibration (artifact),403
2,Fit/Win rubric (artifact),989
3,Few-shot examples,495
4,Requirements,111
5,Evidence pack,1773
6,Job text,1005


CALIBRATION EXAMPLES: real past assessments of the most similar roles, and what actually happened. Use them to calibrate your numbers against outcomes; the posting below is a different role.
[Example 1] Manager, Merchant Operations at Wolt (level band below)
  Scored then: fit 70, win 52, evidence adjacent. Blockers: Influence without formal authority is named explicitly in the requirements, and it is the documented reason he lost the Wolt panel (argue); Merchant and B2B account operations is a domain he has not worked in (argue); Much of the role is maintaining and distributing recurring reporting, which is below the mandate he is targeting (argue)
  Who beats him: An internal Wolt or Delivery Hero merchant-operations manager who already knows the markets and tooling; the role is largely reporting cadence and follow-up, where domain familiarity beats depth of method.
  What actually happened: applied, rejected (stage reached not recorded)
[Example 2] S&O Manager, Support Tooling and S

True

## Step 11 · Full assessment: the same call the app makes
The output schema lists evidence fields before the scores, so the model fills in requirement verdicts, strengths, gaps and blockers before it writes Fit and Win.

In [14]:
t = time.time(); result = assess.assess(wolt); assess_s = round(time.time() - t, 1)
a = result["assessment"]
wolt_job = {**assess.to_board(result, wolt), "status": "new", "sub_status": None, "applied_on": None, "last_activity": None}
show_html(ui.card(wolt_job)); show_html(ui.how_scored(result))
print(json.dumps({k: a[k] for k in ("fit", "win", "evidence", "disciplineFit", "levelBand", "winWhy", "recommendation")}, indent=1, ensure_ascii=False))
check("11 Assess", "Full RAG + LLM pipeline ran (not the offline fallback)", result["source"] == "rag+llm", f"{assess_s}s · {result['timings']}")
check("11 Assess", "Fit and Win are valid 0-100 scores", all(isinstance(a[k], int) and 0 <= a[k] <= 100 for k in ("fit", "win")), f"fit {a['fit']}, win {a['win']}")
check("11 Assess", "Every requirement received a verdict", {m["requirement_id"] for m in a["requirement_matches"]} >= {q["id"] for q in result["posting"]["requirements"]})

#,Requirement,Verdict,Evidence,Note
R1,Own and drive a defect elimination program end-to-end across multiple markets.critical · P1,direct,"E1, E3","He has owned global post-delivery defect areas and describes the end-to-end method of identifying, prioritising and driving defects to resolution across partner teams."
R2,Have direct experience in customer experience or operations translating defects into root-cause fixes.critical · Domain,direct,"E3, E4","Eight years in Amazon Customer Experience and operations translating contacts, defects, concessions and fraud signals into root-cause fixes."
R3,Bring prior experience with a successful voice of the customer program.critical · Domain,adjacent,"E1, E3, E4","His career is built around Voice of the Customer through contact drivers, defect elimination and customer-support signals, but the evidence does not name a formal programme explicitly titled VoC."
R4,Use strong analytical judgment to work with imperfect data and build a credible case from directional evidence.important · P3,direct,"E5, E6","He built conservative, finance-validated cases from directional data and used regression, CART and labelled-contact ground truth to separate controllable from non-controllable variance."
R5,Influence cross-functional partners to prioritize and deliver customer fixes.important · P4,direct,"E7, E8",He influenced five internal teams plus Apple and DHL without formal authority and secured concrete controls that closed an $85M fraud mechanism.
R6,"Prepare and present clear insights in reviews, planning forums, and leadership meetings.important · P4",direct,"E9, E10",He created an adopted value framework used in investment decisions and rebuilt inaccurate reporting used by leadership.
R7,Maintain and improve reporting or data infrastructure that makes customer experience insights actionable.important · P2,adjacent,"E6, E10, E11","He has materially improved measurement, reporting and audit infrastructure, including a 93.5%-accurate tool and corrected contact-reason reporting; direct ownership of Wolt-style CX data infrastructure is not fully evidenced."
Id,Source,For,Score,Excerpt
E1,Ayhan Demir Wolt 35 HM Questions Concise4. What do you understand this role to be?,"R1, R2, R3, R7",3.768,"Defect elimination across 40 EMEA markets, Wolt and Deliveroo. The core is turning customer-experience signals into a cost-ranked defect list using the driver taxonomy, then driving fixes through partner teams in enginee…"
E2,"Ayhan Demir Wolt Dan CoreProjectsQuestion 6: This is your first marketplace. What worries you, and how would you close it?",R1,-0.509,"Situation. I have not worked in a food-delivery or two-sided marketplace, so the specific defect types are new. Task. Close the domain gap fast while applying the domain-independent defect-elimination method I already h…"


{
 "fit": 84,
 "win": 54,
 "evidence": "direct",
 "disciplineFit": "strong",
 "levelBand": "senior-mgr",
 "winWhy": "He is a strong substantive match and would likely be read seriously or shortlisted, with direct evidence on defect elimination, imperfect data and cross-functional delivery. However, this is a deep pool for a Wolt senior programme role, and his prior Wolt final-round loss on executive stakeholder management, influencing without authority and incomplete-data decisiveness is directly relevant. A candidate already running defect elimination in food delivery or a two-sided marketplace, especially one with visibly stronger executive presence, is the specific profile most likely to beat him.",
 "recommendation": "Pursue strongly, but treat it as a competitive re-application: the functional fit is excellent, while the application and interviews must explicitly close the marketplace-context and executive-influence concerns that previously separated him from the winner."
}
✅ Full

True

## Step 12 · Grounding and guardrails
Citations are checked against the evidence pack. The rubric's hard rules are enforced in code, not left to the model, and are unit-tested here on edge cases.

In [15]:
ids = {e["eid"] for e in result["evidence"]}
points = a["strong"] + a["reframeable"]
cited = [p for p in points if p["evidence_ids"]]
display(pd.DataFrame([{"field": k, "LLM raw": result["raw"][k], "after guardrails": a[k]} for k in ("fit", "win")]))
print("Guardrail notes on this run:", result["guardrails"] or "none needed")
check("12 Grounding", "Every cited evidence id exists in the pack", all(e in ids for p in points for e in p["evidence_ids"]))
check("12 Grounding", "At least 80% of strong and reframeable points cite evidence", len(cited) / max(1, len(points)) >= 0.8, f"{len(cited)}/{len(points)} cited")

base = {"strong": [], "reframeable": [], "requirement_matches": [], "germanRequired": False}
g1, n1 = scoring.guard({**base, "fit": 90, "win": 72, "gaps": ["a", "b"], "evidence": "direct", "germanRequired": True, "blockers": []}, set())
g2, n2 = scoring.guard({**base, "fit": 70, "win": 80, "gaps": [], "blockers": [], "evidence": "argued",
                        "strong": [{"text": "x", "evidence_ids": ["E1", "E99"]}]}, {"E1"})
s = scoring.sanitize({"winWhy": "Two promotions, L5 to L6, and he authored the ACES AI Strategy 2026. AuditAI hit 93.5%.", "pillar": "VP WW Global Programs"})
display(pd.DataFrame([{"case": "German required, fit 90 with 2 gaps, win 72", "result": f"fit {g1['fit']}, win {g1['win']}", "notes": " | ".join(n1)},
                      {"case": "argued evidence, win 80, citation E99 not in pack", "result": f"win {g2['win']}, cites {g2['strong'][0]['evidence_ids']}", "notes": " | ".join(n2)},
                      {"case": "sanitiser input with level codes, excluded project, VP title", "result": f"{s['winWhy']} / {s['pillar']}", "notes": ""}]))
check("12 Guardrails", "German requirement adds a screen that caps Win at 25; 85+ Fit needs at most one gap", g1["win"] == 25 and g1["fit"] == 84)
check("12 Guardrails", "Non-direct evidence caps Win at 70 and invented citations are dropped", g2["win"] == 70 and g2["strong"][0]["evidence_ids"] == ["E1"])
check("12 Guardrails", "Sanitiser strips level codes, the excluded project and the VP title", "L6" not in s["winWhy"] and "ACES AI" not in s["winWhy"] and "VP" not in s["pillar"])

,field,LLM raw,after guardrails
0,fit,86,84
1,win,54,54


Guardrail notes on this run: ['Fit 86 capped at 84: 85+ allows at most one minor gap, found 4.']
✅ Every cited evidence id exists in the pack
✅ At least 80% of strong and reframeable points cite evidence  (10/10 cited)


,case,result,notes
0,"German required, fit 90 with 2 gaps, win 72","fit 84, win 25",Added a German-language screen because the posting requires fluent German. | Win 72 capped at 25: hard scr...
1,"argued evidence, win 80, citation E99 not in pack","win 70, cites ['E1']","Removed citation(s) E99 that are not in the evidence pack. | Win 80 capped at 70: best evidence is argued,..."
2,"sanitiser input with level codes, excluded project, VP title",AuditAI hit 93.5%. / Worldwide Programme Lead,


✅ German requirement adds a screen that caps Win at 25; 85+ Fit needs at most one gap
✅ Non-direct evidence caps Win at 70 and invented citations are dropped
✅ Sanitiser strips level codes, the excluded project and the VP title


True

## Step 13 · Calibration against your own history
The artifact board scored this role after you had reached the final round (Fit 86, Win 78). This pipeline reads the posting alone and leaves that entry out, so a Win near the rubric's posting-only ceiling of 70 is the expected, calibrated result.

In [16]:
hist = next(j for j in json.loads(config.HISTORY_PATH.read_text()) if j["company"] == "Wolt" and "Defect Elimination" in j["title"])
display(pd.DataFrame([{"scored by": "Artifact board (after the final round)", "fit": hist["ai"]["score"], "win": hist["ai"]["win"], "outcome": assess.outcome(hist)},
                      {"scored by": "This pipeline (posting only, leave-one-out)", "fit": a["fit"], "win": a["win"], "outcome": "n/a"}]))
check("13 Calibration", "Fit agrees with the original board within 10 points", abs(a["fit"] - hist["ai"]["score"]) <= 10, f"{a['fit']} vs {hist['ai']['score']}")
check("13 Calibration", "Win respects the posting-only ceiling (≤ 70)", a["win"] <= 70, f"win {a['win']}; the board's 78 came after a final round")

,scored by,fit,win,outcome
0,Artifact board (after the final round),86,78,"applied, rejected after reaching panel / final"
1,"This pipeline (posting only, leave-one-out)",84,54,n/a


✅ Fit agrees with the original board within 10 points  (84 vs 86)
✅ Win respects the posting-only ceiling (≤ 70)  (win 54; the board's 78 came after a final round)


True

## Step 14 · Negative control
This synthetic posting (a test fixture, fictional employer) breaks the profile on discipline, language and location: embedded software engineering, C2 German, on-site in Stuttgart.

In [17]:
NEGATIVE = (
    "Senior Embedded Software Engineer (m/w/d) - Steuergeräte-Software\n"
    "Beispiel Automotive GmbH · Stuttgart, Deutschland · Vor Ort\n"
    "Ihre Aufgaben: Sie entwickeln Embedded-Software in C und C++ für Motorsteuergeräte nach AUTOSAR, entwerfen die Software-Architektur, "
    "implementieren Treiber und testen auf Hardware-in-the-Loop-Prüfständen.\n"
    "Ihr Profil: Abgeschlossenes Studium der Informatik oder Elektrotechnik, mindestens 5 Jahre Erfahrung in der Embedded-Entwicklung "
    "(C, C++, AUTOSAR, MISRA). Tägliche Programmierung ist der Kern der Rolle. Verhandlungssichere Deutschkenntnisse (C2) und gute Englischkenntnisse.\n"
    "Wir bieten: Flexible Arbeitszeiten und betriebliche Altersvorsorge.")
neg = assess.assess(NEGATIVE); na = neg["assessment"]
show_html(ui.card({**assess.to_board(neg, NEGATIVE), "status": "new"}))
print("Guardrail notes:", neg["guardrails"] or "none needed")
check("14 Negative control", "Software-engineering role with C2 German gets a low Fit", na["fit"] < 40, f"fit {na['fit']}")
check("14 Negative control", "Hard screens hold Win at 25 or below", na["win"] is not None and na["win"] <= 25, f"win {na['win']}")
check("14 Negative control", "German requirement recognised as a screen", na["germanRequired"] and any(b["kind"] == "screen" for b in na["blockers"]))

Guardrail notes: none needed
✅ Software-engineering role with C2 German gets a low Fit  (fit 4)
✅ Hard screens hold Win at 25 or below  (win 2)
✅ German requirement recognised as a screen


True

## Step 15 · Offline fallback
When no LLM is reachable, the app falls back to the artifact's keyword scorer, with the same bucket weights and gates.

In [18]:
off_w, off_n = assess.assess(wolt, use_llm=False), assess.assess(NEGATIVE, use_llm=False)
display(pd.DataFrame([{"posting": "Wolt Senior PM", "offline fit": off_w["assessment"]["fit"], "RAG+LLM fit": a["fit"], "RAG+LLM win": a["win"]},
                      {"posting": "Embedded engineer (negative control)", "offline fit": off_n["assessment"]["fit"], "RAG+LLM fit": na["fit"], "RAG+LLM win": na["win"]}]))
check("15 Fallback", "Offline scorer works without any LLM", off_w["source"] == "offline" and off_w["assessment"]["fit"] is not None)
check("15 Fallback", "Offline scorer still ranks the right role above the negative control", off_w["assessment"]["fit"] > off_n["assessment"]["fit"],
      f"{off_w['assessment']['fit']} vs {off_n['assessment']['fit']}")

,posting,offline fit,RAG+LLM fit,RAG+LLM win
0,Wolt Senior PM,42,84,54
1,Embedded engineer (negative control),0,4,2


✅ Offline scorer works without any LLM
✅ Offline scorer still ranks the right role above the negative control  (42 vs 0)


True

## Step 16 · Job board database (SQLite)
The board starts with the 43 scored jobs exported from the Claude artifact. Ids use the artifact's own scheme, so re-scoring a known job updates its row and keeps its pipeline stage.

In [19]:
store.init(); store.import_history()
wolt_id = store.upsert(assess.to_board(result, wolt)); neg_id = store.upsert(assess.to_board(neg, NEGATIVE))
jobs = store.jobs(); wolt_row = store.get(wolt_id)
display(pd.DataFrame(jobs).groupby(["status", "source"]).size().rename("jobs").reset_index())
rows, unknown = store.funnel()
display(pd.DataFrame(rows, columns=["Stage reached", "Applications"]))
history_ids = {j["id"] for j in json.loads(config.HISTORY_PATH.read_text())}
check("16 Board DB", "All 43 artifact jobs are on the board", history_ids <= {j["id"] for j in jobs}, f"{len(jobs)} jobs on the board")
check("16 Board DB", "Re-scoring Wolt updated its existing row and kept its stage", wolt_id in history_ids and wolt_row["status"] == "saved" and wolt_row["sub_status"] == "rejected",
      f"id {wolt_id} · {wolt_row['status']} / {wolt_row['sub_status']}")
check("16 Board DB", "New assessments are stored with their RAG trace", bool(wolt_row["ai"]["rag"]["evidence"]), f"{len(wolt_row['ai']['rag']['evidence'])} evidence chunks stored")

,status,source,jobs
0,dismissed,artifact-history,19
1,new,rag+llm,1
2,saved,artifact-history,23
3,saved,rag+llm,1


,Stage reached,Applications
0,Applied,24
1,Recruiter screen,3
2,Hiring manager,2
3,Assignment / case,1
4,Panel / final,1
5,Offer,0


✅ All 43 artifact jobs are on the board  (44 jobs on the board)
✅ Re-scoring Wolt updated its existing row and kept its stage  (id j1hmn0gk · saved / rejected)
✅ New assessments are stored with their RAG trace  (11 evidence chunks stored)


True

## Step 17 · Application materials
This is the Lovable app's materials tab, rebuilt on RAG. The draft is grounded in the canonical record plus this job's evidence pack, styled on your most similar real cover letter (found by metadata-filtered search), and checked against 11 drafting rules. If any rule fails, one self-repair round runs.

In [20]:
t = time.time(); out = materials.generate(store.get(wolt_id)); mat_s = round(time.time() - t, 1)
display(Markdown("**Cover letter**\n\n" + out["materials"]["coverLetter"]))
print(materials.plain_cv(out["materials"])[:2200], "\n…")
display(pd.DataFrame(out["checks"]))
first_failed = [c["check"] + (f" ({c['detail']})" if c["detail"] else "") for c in out["first_draft_checks"] if not c["pass"]]
print("First draft broke:", first_failed or "nothing", "| self-repair round used:", out["repaired"])
print("Style example:", out["style_example"], f"| {mat_s}s | files:", [Path(f).name for f in out["files"]])
check("17 Materials", "Cover letter and CV drafted", bool(out["materials"]["coverLetter"]) and bool(out["materials"]["cv"]["experience"]))
check("17 Materials", "Final draft passes every drafting rule", all(c["pass"] for c in out["checks"]),
      f"{sum(c['pass'] for c in out['checks'])}/{len(out['checks'])} rules" + (f"; first draft fixed by self-repair: {'; '.join(first_failed)}" if first_failed else ""))
check("17 Materials", "Word files exported", all(Path(f).exists() for f in out["files"]), ", ".join(Path(f).name[:42] for f in out["files"][:2]))

**Cover letter**

Dear Wolt Hiring Team,

Every contact that reaches a support agent represents a defect somewhere in the system. That principle has guided my eight years in Amazon Customer Service, where I translated customer signals into prioritised defect lists, built cases from imperfect data, and drove systemic fixes with teams outside my reporting line.

My career moved from engineering and manufacturing project delivery at Total Teknik and Arcelik to global customer experience and process excellence at Amazon. That combination has made me comfortable moving between operational detail, analytical evidence and leadership decisions.

At Amazon, the Product Life Support (PLS) Multi-Phase Programme applied a repeatable defect-elimination method across post-delivery customer issues. It produced $15MM annualised impact with a 45% Service Level Agreement gain. Across eight years, my work delivered $147MM+ confirmed and finance-validated total impact. This included the $132M Concession Permission Clean-Up through access governance and the closure of $85M Mail Label Fraud exposure, coordinated across five internal functions plus Apple and DHL.

I have also built the measurement needed to make customer experience action-oriented. AuditAI reached 93.5% validated accuracy across 10,000 contacts and cut the audit Service Level Agreement 70%, from 7 days to 2. Separately, a Back-Up Work Group (BUWG) Kaizen improved activation Service Level Agreement performance by 80%. These were distinct efforts, both grounded in clearer defect classification and practical follow-through.

The formal name was not Voice of the Customer, but the substance was: contact-driver signals became root-cause analysis, a cost-ranked backlog and tracked partner fixes. I have not worked in food delivery or a two-sided marketplace. I would close that domain gap through early contact-data analysis and by attributing each defect to the courier, merchant or customer marketplace side.

I would welcome the opportunity to discuss how this experience could support defect elimination across Wolt and Deliveroo markets.

Sincerely,
Ayhan Demir

Ayhan Demir
Process Excellence Senior Advisor | Lean Six Sigma Black Belt | Customer Experience and Defect Elimination
Berlin, Germany · ayhandemir7@gmail.com

PROFESSIONAL SUMMARY
Process Excellence Senior Advisor with 19 years of experience across global customer experience, operations, manufacturing and engineering. Eight years at Amazon Customer Excellence System (ACES), translating customer contact signals into root-cause fixes and cross-functional delivery, with $147MM+ confirmed and finance-validated total impact. Experienced in Voice of the Customer work in substance, analytical judgment with imperfect data, reporting improvement and influence without formal authority. German and EU citizen, dual with Turkish nationality. Hybrid preferred, remote EU acceptable.

PROFESSIONAL EXPERIENCE
Process Excellence Senior Advisor, WW CS ACES, Amazon EU/WW Customer Service  (2018 to 2026)  Berlin, Germany
- Delivered $147MM+ confirmed and finance-validated total impact across eight years t

,check,pass,detail
0,Cover letter is 250 to 330 words,True,303 words
1,No em or en dashes,True,
2,No banned phrases,True,
3,Excluded project never mentioned,True,
4,No 'VP' for the wellbeing role,True,
5,$2.1B / $8B only with analysis verbs,True,
6,Every number is in the sources,True,
7,Each locked figure sits with its own project,True,
8,"At most 4 roles, 5 bullets each",True,3 roles
9,Education populated,True,


First draft broke: nothing | self-repair round used: False
Style example: Wolt Cx Cover letter | 18.5s | files: ['Ayhan_Demir_CV_Wolt_Senior_Program_Manager_Customer_Experience_and_Defect_E.docx', 'Ayhan_Demir_CoverLetter_Wolt_Senior_Program_Manager_Customer_Experience_and_Defect_E.docx', 'Ayhan_Demir_Materials_Wolt_Senior_Program_Manager_Customer_Experience_and_Defect_E.md']
✅ Cover letter and CV drafted
✅ Final draft passes every drafting rule  (11/11 rules)
✅ Word files exported  (Ayhan_Demir_CV_Wolt_Senior_Program_Manager, Ayhan_Demir_CoverLetter_Wolt_Senior_Progra)


True

## Step 18 · Gradio app smoke test
The app is launched headless, and its endpoints are called through `gradio_client` the same way the browser calls them.

In [21]:
import app
from gradio_client import Client
demo = app.build()
demo.launch(prevent_thread_lock=True, server_port=7861, theme=ui.theme(), css=ui.CSS, quiet=True)
client = Client("http://127.0.0.1:7861/", verbose=False)
endpoints = sorted(client.view_api(return_format="dict", print_info=False)["named_endpoints"])
kb = client.predict("access governance clean-up with finance validation", "hybrid+rerank", api_name="/kb_search")
links = client.predict("Tier 1", "", "Last week", api_name="/linkedin")
scored = client.predict(NEGATIVE, False, "All", "Any tier", 0, "Fit", api_name="/score")
demo.close()
print("Named endpoints:", endpoints)
print("Top knowledge-base hit through the UI:", (kb[0].get("data") if isinstance(kb[0], dict) else kb[0])[0][:4])
check("18 App", "Gradio app builds and serves its API", {"/score", "/kb_search", "/draft", "/analysis", "/linkedin"} <= set(endpoints), ", ".join(endpoints))
check("18 App", "Knowledge-base search works through the UI", "Concession" in str(kb[0]))
check("18 App", "LinkedIn launcher builds a Berlin, newest-first search", "linkedin.com/jobs/search" in links and "geoId=106967730" in links)
check("18 App", "Score & add works through the UI and returns a job card", "jm-card" in scored[0])

Closing server running on port: 7861
Named endpoints: ['/analysis', '/draft', '/kb_search', '/linkedin', '/rebuild_index', '/score', '/update_job']
Top knowledge-base hit through the UI: [1, 5.1235, 'Ayhan Demir Phase1A Case Studies FINAL June2026', 'Case Study 02 · Concession Permission Clean-Up (EU7) · 2023 · P2 · Portfolio Design and Delivery']
✅ Gradio app builds and serves its API  (/analysis, /draft, /kb_search, /linkedin, /rebuild_index, /score, /update_job)
✅ Knowledge-base search works through the UI
✅ LinkedIn launcher builds a Berlin, newest-first search
✅ Score & add works through the UI and returns a job card


True

## Step 19 · Secret hygiene

In [22]:
leaks, scanned = [], 0
for p in ROOT.rglob("*"):
    if p.is_file() and "chroma" not in p.parts and p.suffix in {".py", ".md", ".json", ".jsonl", ".txt", ".ipynb", ".example", ".csv"}:
        scanned += 1
        text = p.read_text(errors="ignore")
        if any(s in text for s in secrets) or re.search(r"sk-[A-Za-z0-9_-]{30,}", text):
            leaks.append(p.name)
ignored = (ROOT / ".gitignore").read_text()
check("19 Security", "No API key value in any project file", not leaks, f"{scanned} files scanned" if not leaks else leaks)
check("19 Security", "Personal data, the database and secrets are gitignored", all(x in ignored for x in ("data/", ".env", "credentials.json", "token.json")))

✅ No API key value in any project file  (33 files scanned)
✅ Personal data, the database and secrets are gitignored


True

## Summary

In [23]:
summary = pd.DataFrame(CHECKS)
passed = int((summary.Result == "PASS").sum())
display(Markdown(f"### {passed} / {len(summary)} checks passed"))
display(summary)
calls = pd.DataFrame(llm.CALLS)
display(calls)

def md_table(df):
    cols = list(df.columns)
    lines = ["| " + " | ".join(map(str, cols)) + " |", "|" + "---|" * len(cols)]
    lines += ["| " + " | ".join(str(v).replace("|", "/").replace("\n", " ") for v in row) + " |" for row in df.itertuples(index=False)]
    return "\n".join(lines)

minutes = round((time.time() - RUN_STARTED) / 60, 1)
verdicts = pd.DataFrame([{"Req": m["requirement_id"], "Verdict": m["verdict"], "Evidence": ", ".join(m["evidence_ids"]), "Note": m["note"][:110]}
                         for m in a["requirement_matches"]])
proof = f'''# Lab proof · Job Match Studio

Executed notebook: [`lab_proof.ipynb`](lab_proof.ipynb) · run {time.strftime("%Y-%m-%d %H:%M")} · {minutes} min ·
LLM `{config.LLM_MODEL}` (decomposition `{config.LLM_MODEL_FAST}`) · embeddings `{config.EMBED_MODEL}` (local) · reranker `{rag.reranker_name()}`

**Result: {passed} / {len(summary)} checks passed.**

## Checks

{md_table(summary)}

## Retrieval evaluation (21 labelled queries: 12 keyword-rich, 9 paraphrased)

{md_table(EVAL_TABLE)}

## Real posting: Wolt, Senior Program Manager CX and Defect Elimination

- This pipeline: **Fit {a["fit"]} · Win {a["win"]}** · evidence {a["evidence"]} · {a["disciplineFit"]} discipline fit
- Artifact board, scored after the final round: Fit {hist["ai"]["score"]} · Win {hist["ai"]["win"]} · outcome: {assess.outcome(hist)}
- Who beats him: {a["winWhy"]}
- Recommendation: {a["recommendation"]}

{md_table(verdicts)}

## Negative control (embedded software, C2 German, on-site Stuttgart)

Fit {na["fit"]} · Win {na["win"]} · screens: {"; ".join(b["text"] for b in na["blockers"] if b["kind"] == "screen")}

## Application materials checks

{md_table(pd.DataFrame(out["checks"]))}

## LLM calls in this run

{md_table(calls)}

## Screenshots of the running app

Captured from the live Gradio app (Playwright driving Chrome).

| Board: Fit/Win gauge cards | Card back: full analysis with cited evidence |
|---|---|
| ![Board](screenshots/board_cards.png) | ![Card](screenshots/card_full_analysis.png) |

| Analysis tab | Knowledge Base tab: retrieval playground |
|---|---|
| ![Analysis](screenshots/analysis_tab.png) | ![Knowledge base](screenshots/knowledge_base_tab.png) |

## Reproduce

```bash
conda activate bootcamp-env && cd job-search-score-apply-bot
export OPENAI_API_KEY=...  COHERE_API_KEY=...   # or keep them in ~/.config/job-search-bot/.env, never inside the project
python -m jupyter nbconvert --to notebook --execute quality_check/lab_proof.ipynb --inplace
```
'''
(ROOT / "quality_check" / "LAB_PROOF.md").write_text(proof, encoding="utf-8")
print("Wrote quality_check/LAB_PROOF.md")

### 60 / 60 checks passed

,Step,Check,Result,Evidence
0,0 Setup,OpenAI key available to this run (value never printed),PASS,
1,0 Setup,"Cohere key available, so reranking uses rerank-v3.5",PASS,cohere
2,0 Setup,"Keys come from the environment, not from a file in the project",PASS,
3,1 Drive,Core project documents mirrored from Google Drive,PASS,"11 documents, 179,226 characters"
4,1 Drive,Every mirrored document has text,PASS,"smallest has 1,924 characters"
5,1 Drive,Each document links back to its Drive file,PASS,
6,1 Drive,Sync keeps only the newest version of each document,PASS,3 of 12 kept
7,1 Drive,"Sync skips backups, copies, images, code exports and oversized scans",PASS,
8,2 Canonicalise,Excluded project (ACES AI Strategy) removed from the index,PASS,"16 mentions in the Phase 0 map, 0 in the index"
9,2 Canonicalise,Canonical title enforced (no 'VP' for the wellbeing role),PASS,10 rewrites


,model,schema,input_tokens,output_tokens,seconds
0,gpt-5.4-mini,Posting,1432,537,4.1
1,gpt-5.4-mini,Posting,1432,555,4.1
2,gpt-5.6-luna,Assessment,7252,2015,24.1
3,gpt-5.4-mini,Posting,597,572,4.0
4,gpt-5.6-luna,Assessment,6843,1466,18.2
5,gpt-5.6-luna,Materials,6161,1977,18.4
6,gpt-5.4-mini,Posting,597,429,3.1
7,gpt-5.6-luna,Assessment,6084,1444,17.5


Wrote quality_check/LAB_PROOF.md
